# KnowledgeHub RAG v0.6.4 - Context Assembly Fix + Full Engineering Checklist

## Objective

This version fixes two real bugs surfaced by the v0.6.5 eval run, and folds
in the full pre-Streamlit engineering checklist (index persistence, model
caching, latency timing, dependency pinning, error handling) so this is a
single complete version to test against — with only the different-PDF test
remaining afterward.

## RAG-quality fixes (evidence-based, from your v0.6.5 run)

1. **Context-assembly slicing bug, fixed.** `answer_question()` built the
   LLM's context with `retrieved_chunks[:top_context]` - the same kind of
   positional slice we already fixed in `evaluate_retrieval`, except this
   one was still live in the part that actually decides what the model
   sees. Since `retrieved_chunks` is grouped `[rank-1 + its neighbors,
   rank-2 + its neighbors, ...]`, slicing `[:3]` almost always returned
   only rank-1's own neighborhood - so the LLM rarely saw the actual
   content from ranks 2-3, even when they were correctly retrieved. New
   `build_context()` groups by `retrieval_rank` and takes the top-N
   *distinct* ranks instead. This is very likely why "What does Classifier A
   predict?" kept giving vague/incomplete answers despite the right page
   being retrieved.
2. **Query expansion gap fixed.** "Which real gravitational-wave events
   were analysed?" wasn't just a generation hallucination - your v0.6.5
   `evaluate_rag()` run showed it as a genuine retrieval miss under
   hybrid-only too (`[6,4,1]`, gold page 7 absent). `QUERY_EXPANSION` had
   no mapping at all for "event(s)". Added `GW170817`, `GW190425`,
   "gravitational-wave event", "real observation" as expansion terms.
3. **Author-name diagnostic added, not blindly fixed.** Your eval run
   generated "Kannyyappan" (missing an 'i') where every prior run had it
   correct. Rather than paper over this with fuzzy keyword matching (which
   risks masking real errors elsewhere), the diagnostic cell now prints the
   actual source chunk containing the name, so you can see directly whether
   this is a PDF-extraction artifact or a generation slip.

## Engineering checklist, folded in as promised

4. **Index persistence** - FAISS index + chunks now cache to disk (on
   Drive, alongside your data folder), keyed to a fingerprint of the source
   PDFs. Re-running skips re-embedding entirely unless the PDFs changed.
5. **Model-load caching** - LLM and reranker cells now check `if "model"
   not in globals()` before reloading, so re-running a cell mid-session
   doesn't reload multi-GB weights unnecessarily.
6. **Latency timing** - `answer_question()` now returns elapsed time as a
   4th value; printed in interactive chat and eval output, so you have
   real numbers on whether this feels fast enough for a UI before building one.
7. **`requirements.txt`** generated and copied to Drive, pinning the
   environment while everything currently works.
8. **Basic error handling** - `generate_answer()` and `answer_question()`
   wrapped in try/except with graceful fallback messages instead of
   crashing a long eval or interactive session; interactive loop skips
   empty input instead of erroring.

## What did NOT change

- Retrieval defaults to hybrid-only (confirmed correct by your own
  `compare_retrieval_modes()` run: 7/8 vs 6/8 reranked vs 7/8 blended).
  Reranking remains available and opt-in.
- Conversation-aware query rewriting - confirmed working in your v0.6.5
  interactive session ("What algorithm did he use?" now retrieves and
  answers correctly). Unchanged.
- LaTeX-robust matching and the no-LaTeX prompt rule - unchanged, still needed.
- No second/different PDF has been tested. **This is the one remaining
  step before v0.7** - everything else asked for is in this version.


In [ ]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers
!pip install -q accelerate
!pip install -q rank-bm25
!pip install -q langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 2.5 MB/s eta 0:00:00


In [ ]:
import os
import time
import faiss
import numpy as np

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import pipeline

from rank_bm25 import BM25Okapi

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


Find PDFs in the data folder. Wrapped so a misconfigured path fails with a clear message instead of a bare traceback.

In [ ]:
import os

DATA_PATH = "/content/drive/MyDrive/KnowledgeHub_RAG/data"

try:
    pdf_files = [
        os.path.join(DATA_PATH, file)
        for file in os.listdir(DATA_PATH)
        if file.lower().endswith(".pdf")
    ]
except FileNotFoundError:
    pdf_files = []
    print(f"DATA_PATH does not exist: {DATA_PATH}")
    print("Check that Drive is mounted and the folder path is correct.")

print(f"Found {len(pdf_files)} PDF(s):")

for pdf in pdf_files:
    print("-", os.path.basename(pdf))


Found 1 PDF(s):
- Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf


Load PDFs. Each file is wrapped individually so one corrupt/unreadable PDF doesn't take down the whole ingestion run - it's skipped with a clear message instead.

In [ ]:
def load_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        extracted = page.extract_text()

        if extracted:

            pages.append(
                {
                    "page": page_number,
                    "text": extracted
                }
            )

    return pages


documents = []

for pdf in pdf_files:

    try:
        pages = load_pdf(pdf)

        documents.append(
            {
                "filename": os.path.basename(pdf),
                "pages": pages
            }
        )

    except Exception as e:
        print(f"Skipping {os.path.basename(pdf)} - failed to load: {type(e).__name__}: {e}")

print(f"Loaded {len(documents)} document(s).")


Loaded 1 document(s).


In [ ]:
!pip install -q langchain-text-splitters

Pin the environment now, while everything currently works. A copy goes to
Drive so it survives a Colab runtime reset, not just the local `/content` disk.

In [ ]:
!pip freeze > requirements.txt

_drive_req_path = os.path.join(os.path.dirname(DATA_PATH), "requirements.txt")

try:
    import shutil
    shutil.copy("requirements.txt", _drive_req_path)
    print(f"Saved requirements.txt (local) and copied to {_drive_req_path}")
except Exception as e:
    print(f"Saved requirements.txt locally. Drive copy skipped: {type(e).__name__}: {e}")


Saved requirements.txt (local) and copied to /content/drive/MyDrive/KnowledgeHub_RAG/requirements.txt


In [ ]:
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter


# ==========================================================
# Heading Detection
# ==========================================================

def is_heading(line):

    line = line.strip()

    if len(line) < 2:
        return False

    # 1
    # 1.2
    # 2.3.4
    if re.match(r"^\d+(\.\d+)*\s+[A-Z]", line):
        return True

    # ALL CAPS
    if line.isupper() and len(line.split()) <= 8:
        return True

    # Markdown
    if line.startswith("#"):
        return True

    # Very short title
    if len(line.split()) <= 8 and line.endswith(":"):
        return True

    return False


# ==========================================================
# Split page into sections using headings
# ==========================================================

def split_into_sections(text):

    lines = text.split("\n")

    sections = []

    current = []

    for line in lines:

        if is_heading(line):

            if current:
                sections.append("\n".join(current).strip())

            current = [line]

        else:

            current.append(line)

    if current:
        sections.append("\n".join(current).strip())

    return sections


# ==========================================================
# Recursive splitter
# ==========================================================

splitter = RecursiveCharacterTextSplitter(

    chunk_size=800,

    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)


# ==========================================================
# Final Chunking Function
# ==========================================================

def chunk_text(text):

    sections = split_into_sections(text)

    final_chunks = []

    for section in sections:

        if len(section) <= 900:

            final_chunks.append(section)

        else:

            final_chunks.extend(
                splitter.split_text(section)
            )

    return final_chunks

In [ ]:
all_chunks = []

chunk_id = 0

for document in documents:

    for page in document["pages"]:

        sections = split_into_sections(page["text"])

        for section in sections:

            # -----------------------------
            # Extract heading
            # -----------------------------
            lines = section.split("\n")

            heading = ""

            if lines and is_heading(lines[0]):
                heading = lines[0].strip()

            # -----------------------------
            # Split section if needed
            # -----------------------------
            if len(section) <= 900:

                section_chunks = [section]

            else:

                section_chunks = splitter.split_text(section)

            # -----------------------------
            # Save chunks
            # -----------------------------
            for chunk in section_chunks:

                all_chunks.append(
                    {
                        "chunk_id": chunk_id,
                        "document": document["filename"],
                        "page": page["page"],
                        "section": heading,
                        "text": chunk
                    }
                )

                chunk_id += 1

print(f"Created {len(all_chunks)} chunks.")

Created 112 chunks.


In [ ]:
print("=" * 80)

print("FIRST 10 CHUNKS")

print("=" * 80)

for chunk in all_chunks[:10]:

    print(f"\nChunk ID : {chunk['chunk_id']}")

    print(f"Page     : {chunk['page']}")


    print("-" * 80)

    print(chunk["text"][:400])

    print()

FIRST 10 CHUNKS

Chunk ID : 0
Page     : 1
--------------------------------------------------------------------------------
Machine Learning Classification of Binary Neutron
Star Remnants Using Gravitational Wave Data
Surendaranath Kanniyappan
Dr. Michalis Agathos
Abstract
Binary neutron star (BNS) mergers are among the most energetic cosmic events,
producing gravitational waves (GWs), electromagnetic (EM) counterparts, and potentially
neutrinos. These mergers provide an unparalleled opportunity to study supranuclear m


Chunk ID : 1
Page     : 1
--------------------------------------------------------------------------------
hypermassive neutron star (short- or long-lived HMNS), or remains stable.
Direct detection of postmerger GW signals remains challenging due to their high
frequency nature (≳ 1 kHz) and the sensitivity limits of current interferometers. Therefore,
predicting remnant outcomes from inspiral parameters—total mass Mtot, mass ratio q, tidal
deformability ˜Λ, and effecti

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Build or load the FAISS index

**v0.6.6 addition:** caches the FAISS index + chunks to disk (on Drive,
alongside your data folder), fingerprinted against the source PDFs. If
nothing changed, this skips re-embedding entirely - the single change most
likely to make a Streamlit UI feel usable instead of rebuilding the whole
index on every rerun. If you add/change PDFs, the fingerprint mismatch
triggers a clean rebuild automatically.

In [ ]:
import hashlib
import pickle

CACHE_DIR = os.path.join(os.path.dirname(DATA_PATH), "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

FAISS_INDEX_PATH = os.path.join(CACHE_DIR, "faiss.index")
CHUNKS_CACHE_PATH = os.path.join(CACHE_DIR, "chunks.pkl")
FINGERPRINT_PATH = os.path.join(CACHE_DIR, "fingerprint.txt")


def _corpus_fingerprint():
    sig = "|".join(
        sorted(
            f"{os.path.basename(p)}:{os.path.getsize(p)}"
            for p in pdf_files
        )
    )
    return hashlib.md5(sig.encode()).hexdigest()[:16]


def load_index_cache():

    if not (
        os.path.exists(FAISS_INDEX_PATH)
        and os.path.exists(CHUNKS_CACHE_PATH)
        and os.path.exists(FINGERPRINT_PATH)
    ):
        return None, None

    with open(FINGERPRINT_PATH) as f:
        cached_fp = f.read().strip()

    if cached_fp != _corpus_fingerprint():
        print("Cache fingerprint mismatch (source PDFs changed) - rebuilding.")
        return None, None

    cached_index = faiss.read_index(FAISS_INDEX_PATH)

    with open(CHUNKS_CACHE_PATH, "rb") as f:
        cached_chunks = pickle.load(f)

    return cached_index, cached_chunks


def save_index_cache(index_to_save, chunks_to_save):

    faiss.write_index(index_to_save, FAISS_INDEX_PATH)

    with open(CHUNKS_CACHE_PATH, "wb") as f:
        pickle.dump(chunks_to_save, f)

    with open(FINGERPRINT_PATH, "w") as f:
        f.write(_corpus_fingerprint())

    print(f"Saved index cache to {CACHE_DIR}")


cached_index, cached_chunks = load_index_cache()

if cached_index is not None:

    index = cached_index
    all_chunks = cached_chunks

    print(f"Loaded cached FAISS index ({index.ntotal} vectors) and {len(all_chunks)} chunks - skipped re-embedding.")

else:

    texts = [chunk["text"] for chunk in all_chunks]

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)

    index.add(embeddings.astype("float32"))

    print(f"Built new FAISS index: {index.ntotal} vectors")

    save_index_cache(index, all_chunks)


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Built new FAISS index: 112 vectors
Saved index cache to /content/drive/MyDrive/KnowledgeHub_RAG/cache


In [ ]:

tokenized_corpus = [
    chunk["text"].lower().split()
    for chunk in all_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

print(" BM25 Index Built")

 BM25 Index Built


## Load the rerankerKept and loadable, but **off by default** in `retrieve()` after the v0.6.4
eval run showed it losing a previously-correct chunk for a direct factual
question. Use `compare_retrieval_modes()` further down to decide for
yourself whether enabling it helps on this document.Model-load guard added: re-running this cell mid-session won't reload the reranker if it's already in memory.

In [ ]:
from sentence_transformers import CrossEncoder

if "reranker" not in globals():
    print("Loading reranker...")
    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    print("Reranker Loaded (available, but off by default in retrieve())")
else:
    print("Reranker already loaded - skipping reload.")


Loading reranker...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker Loaded (available, but off by default in retrieve())


**v0.6.6 change:** added an `event`/`events` expansion mapping after your
v0.6.5 `evaluate_rag()` run showed "Which real gravitational-wave events
were analysed?" failing retrieval under hybrid-only too (not just
hallucinating during generation) - nothing previously nudged retrieval
toward GW170817/GW190425.

In [ ]:
QUERY_EXPANSION = {

    "algorithm": [
        "model",
        "classifier",
        "GBDT",
        "Gradient Boosted Decision Tree"
    ],

    "gbdt": [
        "Gradient Boosted Decision Tree",
        "gradient boosting"
    ],

    "classifier": [
        "classification model",
        "machine learning model"
    ],

    "accuracy": [
        "performance",
        "evaluation",
        "MCC"
    ],

    "dataset": [
        "training data",
        "simulation dataset"
    ],

    "method": [
        "approach",
        "framework"
    ],

    "event": [
        "GW170817",
        "GW190425",
        "gravitational-wave event",
        "real observation"
    ]

}


def expand_query(query):

    expanded = query

    query_lower = query.lower()

    for key, values in QUERY_EXPANSION.items():

        if key in query_lower:

            expanded += " " + " ".join(values)

    return expanded


In [ ]:
def reciprocal_rank_fusion(semantic_results, bm25_results, k=60):
    """
    Reciprocal Rank Fusion (RRF)

    Score(doc) = Σ 1 / (k + rank)

    Larger k -> smoother scores (60 is the standard value).
    """

    fused = {}

    # Semantic ranking
    for rank, item in enumerate(semantic_results, start=1):
        cid = item["chunk_id"]

        if cid not in fused:
            fused[cid] = item.copy()
            fused[cid]["rrf_score"] = 0.0

        fused[cid]["rrf_score"] += 1.0 / (k + rank)

    # BM25 ranking
    for rank, item in enumerate(bm25_results, start=1):
        cid = item["chunk_id"]

        if cid not in fused:
            fused[cid] = item.copy()
            fused[cid]["rrf_score"] = 0.0

        fused[cid]["rrf_score"] += 1.0 / (k + rank)

    return sorted(
        fused.values(),
        key=lambda x: x["rrf_score"],
        reverse=True
    )

## Retrieve

**v0.6.5 changes:**
- `use_reranker` defaults to `False` - retrieval behaves exactly like the
  proven pre-reranker hybrid pipeline (FAISS + BM25 + RRF) unless you opt in.
- When `use_reranker=True`, the cross-encoder score is **blended** with the
  original RRF-based `final_score` (min-max normalized, weighted by
  `rerank_weight`) rather than fully overriding it - so one cross-encoder
  misjudgment can't completely bury a chunk both hybrid signals agreed on.
  `rerank_weight=1.0` reproduces the old full-override behavior if you want
  to test that exact configuration.

In [ ]:
def _minmax(values):
    lo, hi = min(values), max(values)
    if hi - lo < 1e-9:
        return [0.5 for _ in values]
    return [(v - lo) / (hi - lo) for v in values]


def retrieve(query, top_k=5, rerank_pool_size=20, use_reranker=False, rerank_weight=0.6):

    # =====================================================
    # Encode Query
    # =====================================================
    expanded_query = expand_query(query)

    query_embedding = embedding_model.encode(
        [expanded_query],
        normalize_embeddings=True
    ).astype("float32")

    # =====================================================
    # FAISS Search
    # =====================================================

    semantic_scores, semantic_indices = index.search(
        query_embedding,
        top_k * 8
    )

    semantic_results = []

    for rank, (score, idx) in enumerate(
        zip(semantic_scores[0], semantic_indices[0]),
        start=1
    ):

        semantic_results.append({

            "chunk_id": idx,
            "rank": rank,
            "semantic_score": float(score)

        })

    # =====================================================
    # BM25 Search
    # =====================================================

    tokenized_query = expanded_query.lower().split()

    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_ranked = sorted(

        enumerate(bm25_scores),

        key=lambda x: x[1],

        reverse=True

    )[:top_k * 8]

    bm25_results = []

    for rank, (idx, score) in enumerate(
        bm25_ranked,
        start=1
    ):

        bm25_results.append({

            "chunk_id": idx,
            "rank": rank,
            "bm25_score": float(score)

        })

    # =====================================================
    # Reciprocal Rank Fusion (RRF)
    # =====================================================

    k = 60

    rrf_scores = {}

    semantic_lookup = {}
    bm25_lookup = {}

    for item in semantic_results:

        cid = item["chunk_id"]

        semantic_lookup[cid] = item["semantic_score"]

        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (k + item["rank"])

    for item in bm25_results:

        cid = item["chunk_id"]

        bm25_lookup[cid] = item["bm25_score"]

        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (k + item["rank"])

    rrf_results = []

    for chunk_id, rrf_score in rrf_scores.items():

        chunk = all_chunks[chunk_id]

        rrf_results.append({

            "chunk_id": chunk_id,

            "page": chunk["page"],

            "document": chunk["document"],

            "text": chunk["text"],

            "semantic_score": semantic_lookup.get(chunk_id, 0.0),

            "bm25_score": bm25_lookup.get(chunk_id, 0.0),

            "rrf_score": rrf_score

        })

    # =====================================================
    # Intelligent Tie Breaking
    # =====================================================

    for item in rrf_results:

        item["final_score"] = (

            item["rrf_score"]

            + 0.001 * item["semantic_score"]

            + 0.001 * item["bm25_score"]

        )

    rrf_results = sorted(

        rrf_results,

        key=lambda x: x["final_score"],

        reverse=True

    )

    # =====================================================
    # Cross-Encoder Reranking (OPTIONAL - off by default, see v0.6.5 notes)
    # =====================================================

    if use_reranker and rrf_results:

        pool = rrf_results[:rerank_pool_size]
        remainder = rrf_results[rerank_pool_size:]

        cross_inputs = [(query, item["text"]) for item in pool]
        cross_scores = reranker.predict(cross_inputs)

        for item, score in zip(pool, cross_scores):
            item["cross_score"] = float(score)

        norm_final = _minmax([item["final_score"] for item in pool])
        norm_cross = _minmax([item["cross_score"] for item in pool])

        for item, nf, nc in zip(pool, norm_final, norm_cross):
            item["combined_score"] = (
                rerank_weight * nc + (1 - rerank_weight) * nf
            )

        pool = sorted(pool, key=lambda x: x["combined_score"], reverse=True)

        for item in remainder:
            item["cross_score"] = None

        rrf_results = pool + remainder

    else:
        for item in rrf_results:
            item["cross_score"] = None

    # =====================================================
    # Prevent Adjacent Matched Chunks
    # =====================================================

    selected = []

    for item in rrf_results:

        current = item["chunk_id"]

        if any(abs(current - x["chunk_id"]) <= 1 for x in selected):
            continue

        selected.append(item)

        if len(selected) == top_k:
            break

    # =====================================================
    # Context Expansion
    # =====================================================

    expanded = []

    visited = set()

    for rank, item in enumerate(selected, start=1):

        current = item["chunk_id"]

        for neighbour in [current - 1, current, current + 1]:

            if neighbour < 0:
                continue

            if neighbour >= len(all_chunks):
                continue

            if neighbour in visited:
                continue

            if all_chunks[neighbour]["document"] != item["document"]:
                continue

            visited.add(neighbour)

            chunk = all_chunks[neighbour]

            expanded.append({

                "chunk_id": chunk["chunk_id"],

                "page": chunk["page"],

                "document": chunk["document"],

                "text": chunk["text"],

                "retrieval_rank": rank,

                "context_neighbor": neighbour != current,

                "semantic_score": item["semantic_score"] if neighbour == current else None,

                "bm25_score": item["bm25_score"] if neighbour == current else None,

                "rrf_score": item["rrf_score"] if neighbour == current else None,

                "cross_score": item["cross_score"] if neighbour == current else None

            })

    return expanded


In [ ]:
def debug_retrieval(query, top_k=5):

    results = retrieve(query, top_k)

    print("=" * 90)
    print(f"QUERY : {query}")
    print("=" * 90)

    current_rank = None

    for chunk in results:

        if chunk["retrieval_rank"] != current_rank:

            current_rank = chunk["retrieval_rank"]

            print()
            print("=" * 90)
            print(f"RETRIEVAL RANK {current_rank}")
            print("=" * 90)

        print()

        if chunk["context_neighbor"]:

            print("Context Chunk")

        else:

            print("Matched Chunk")

            print(f"Semantic Score : {chunk['semantic_score']:.4f}")
            print(f"BM25 Score     : {chunk['bm25_score']:.4f}")
            if "rrf_score" in chunk:
              print(f"RRF Score      : {chunk['rrf_score']:.5f}")

        print(f"Document : {chunk['document']}")
        print(f"Page     : {chunk['page']}")

        if chunk.get("section"):
          print(f"Section  : {chunk['section']}")

        print(f"Chunk ID : {chunk['chunk_id']}")

        print("-" * 90)

        print(chunk["text"][:700])

        print()

In [ ]:
debug_retrieval("What algorithm was used for classification?")

debug_retrieval("What is GBDT?")

debug_retrieval("Gradient Boosted Decision Tree")

QUERY : What algorithm was used for classification?

RETRIEVAL RANK 1

Context Chunk
Document : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page     : 8
Chunk ID : 33
------------------------------------------------------------------------------------------
difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more realistic and stable performance estimates.


Matched Chunk
Semantic Score : 0.6347
BM25 Score     : 13.7764
RRF Score      : 0.03

In [ ]:
query = "What algorithm was used for classification?"

results = retrieve(query)

# Keep only the best 2 chunks
context = "\n\n".join(
    [r["text"][:600] for r in results[:2]]
)

print("=" * 80)
print(context)

difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more reali

3.5 Gradient Boosted Decision Tree (GBDT) Training
All three classifiers were trained using the gradient boosting framework for decision trees [27]
as implemented in scikit-learn [28]. This method builds an ensemble of shallow decision
trees (weak learners) sequentially, with each new tree correcting the residual errors of the
previous ensemble. The outputs are combined, weighted by a learning r

In [ ]:
query = "What algorithm was used for classification?"

results = retrieve(query)

print(f"Retrieved {len(results)} chunks.")

Retrieved 15 chunks.


In [ ]:
TOP_CONTEXT = 3

context = "\n\n".join(
    chunk["text"]
    for chunk in results[:TOP_CONTEXT]
)

print("=" * 80)
print(context)

difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more realistic and stable performance estimates.

3.5 Gradient Boosted Decision Tree (GBDT) Training
All three classifiers were trained using the gradient boosting framework for decision trees [27]
as implemented in scikit-learn [28]. This method builds an ensemble of shallow decision
trees (weak learners) sequentially, with each new tree correcting the residual errors of the
previous ensemble. The outputs 

## Diagnostic: document size + author-name check

**v0.6.6 addition:** prints the raw source chunk containing the author's
name. Your v0.6.5 eval run generated "Kannyyappan" (missing an 'i') where
every prior run had it correct - this shows you the actual extracted text
so you can tell whether it's a PDF-extraction artifact (e.g. a hyphenated
line break) or purely a generation slip, instead of guessing.

In [ ]:
print(f"PDFs loaded: {len(pdf_files)}")
for pdf in pdf_files:
    print(" -", pdf)

print(f"\nTotal chunks: {len(all_chunks)}")

pages_seen = sorted(set(c["page"] for c in all_chunks))
print(f"\nPages represented in chunks ({len(pages_seen)} pages): {pages_seen}")

print("\nChunks 18-25 (sample content, if this range exists):")
for chunk in all_chunks[18:26]:
    print("=" * 70)
    print(f"chunk_id: {chunk['chunk_id']}  page: {chunk['page']}  document: {chunk['document']}")
    print(chunk["text"][:200])

print("\n" + "=" * 70)
print("Author-name check (first chunk containing 'kanni'):")
found = False
for chunk in all_chunks:
    if "kanni" in chunk["text"].lower():
        print(f"chunk_id {chunk['chunk_id']} (page {chunk['page']}):")
        print(chunk["text"][:300])
        found = True
        break
if not found:
    print("No chunk contains 'kanni' - check extraction more broadly.")


PDFs loaded: 1
 - /content/drive/MyDrive/KnowledgeHub_RAG/data/Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf

Total chunks: 112

Pages represented in chunks (24 pages): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]

Chunks 18-25 (sample content, if this range exists):
chunk_id: 18  page: 5  document: Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
3.1 Workflow Overview
The classification pipeline developed ingests inspiral-phase parameters from Gravitational
wave data analyses and produces calibrated probabilities for the different possible pos
chunk_id: 19  page: 5  document: Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
(GBDT) models of increasing resolution, as described in Section 3.2.
4. Post-processing: Classifier outputs are converted to calibrated probabilities, with optional
entropy-based thresholding to quant
chunk_id: 20  page: 5  document: Surendaranath_Kanniyappan_FinalProjectReport_SPC720

## Load the LLM

Unchanged from v0.6.3/v0.6.4: `Qwen/Qwen2.5-3B-Instruct`.

Model-load guard added: re-running this cell mid-session won't reload the ~3B-parameter model if it's already in memory.

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

import torch

# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # lighter fallback
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

if "model" not in globals() or "tokenizer" not in globals():

    print("Loading tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Loading model...")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    model.generation_config.max_length = None
    model.generation_config.pad_token_id = tokenizer.pad_token_id

    print(f"{MODEL_NAME} Loaded")

else:
    print(f"{MODEL_NAME} already loaded - skipping reload.")


Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-3B-Instruct Loaded


## Build the prompt

**v0.6.5 additions to the rules:**
- No LaTeX/Markdown math notation - fixes false generation failures where
  the answer was correct but formatted as `\chi_{\text{eff}}` instead of
  plain "χeff".
- When asked what something predicts/classifies, name the output
  categories, not just the inputs - targets the Classifier A answer that
  described its input parameters instead of the black-hole-vs-remnant
  output classes.

In [ ]:
def build_system_prompt(context):

    history = get_chat_history()

    return f"""You are an AI assistant answering questions from a research report.

Rules:

- Answer ONLY using the Context.
- Do NOT use outside knowledge.
- Do NOT invent facts or names.
- Write mathematical symbols and variable names in plain text (for example,
  write "Mtot" and "chi_eff"). Never use LaTeX or Markdown math notation
  such as \\( \\), $, \\text{{}}, or underscores/braces for subscripts.
- If asked what something predicts or classifies, state the specific output
  categories or classes it distinguishes between - not just its inputs.
- If the answer is not present in the context, reply exactly:

I could not find that information in the provided document.

- Keep answers concise.
- Maximum 3 sentences.

Previous Conversation:
{history}

Context:
{context}"""


In [ ]:
# ----------------------------
# Conversation Memory
# ----------------------------

conversation_history = []


def get_chat_history(max_turns=3):
    """
    Returns the previous conversation history
    as formatted text.
    """

    if len(conversation_history) == 0:
        return ""

    history = ""

    for q, a in conversation_history[-max_turns:]:

        history += f"User: {q}\n"
        history += f"Assistant: {a}\n\n"

    return history

**v0.6.6 change:** wrapped in try/except so one malformed generation call doesn't crash a long eval run or interactive session - returns a graceful fallback message instead.

In [ ]:
def generate_answer(question, context):

    try:

        system_prompt = build_system_prompt(context)

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            do_sample=False,
            repetition_penalty=1.15,
            max_new_tokens=150,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

        answer = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        stop_words = [
            "\nQuestion:",
            "\nContext:",
            "\nUser:",
            "\nAssistant:"
        ]

        for stop in stop_words:
            if stop in answer:
                answer = answer.split(stop)[0].strip()

        return answer

    except Exception as e:
        print(f"[generate_answer error] {type(e).__name__}: {e}")
        return "Sorry, I ran into an error generating a response."


Quick raw-generation smoke test (bypasses the confidence gate and query rewriting - direct model check).

In [ ]:
query = "What algorithm was used for classification?"

results = retrieve(query)

context = "\n\n".join(
    chunk["text"]
    for chunk in results[:3]
)

answer = generate_answer(query, context)

print("=" * 80)
print("QUESTION")
print(query)

print("\n" + "=" * 80)
print("ANSWER")
print(answer)

QUESTION
What algorithm was used for classification?

ANSWER
Gradient Boosted Decision Trees (GBDT) were used for classification.


## Retrieval Confidence Threshold ("Don't Know" Mode) + Context Assembly + Query Rewriting

**v0.6.6 changes:**
- **`build_context()` fix (the main one)**: assembles context from the
  top-N *distinct* retrieval ranks instead of positionally slicing the
  expanded (rank+neighbors) list - see the top-of-notebook explanation for
  why this mattered.
- **`answer_question()` now returns `(answer, confidence, retrieved_chunks,
  elapsed_seconds)`** - a 4th value for latency visibility - and is
  wrapped in try/except so a single bad query returns a graceful fallback
  instead of crashing a long session.
- `CONFIDENCE_THRESHOLD` stays at 0.35 - your v0.6.5 calibration run
  confirmed it: unanswerable max 0.301, answerable min 0.411, real gap
  confirmed on real data.

In [ ]:
import math
import time

# Confirmed against your v0.6.5 calibration run: unanswerable max 0.301,
# answerable min 0.411 - 0.35 sits cleanly in that gap on real data.
# Re-run calibrate_confidence_threshold() if you change documents or
# enable use_reranker (different score scale, needs recalibration).
CONFIDENCE_THRESHOLD = 0.35

DONT_KNOW_MESSAGE = "I could not find that information in the provided document."


def assess_confidence(retrieved_chunks):
    """
    Prefers the cross-encoder score (sigmoid-squashed) when present
    (use_reranker=True was used); falls back to embedding semantic_score
    otherwise (the default hybrid-only path).
    """

    cross_scores = [
        chunk["cross_score"]
        for chunk in retrieved_chunks
        if chunk.get("cross_score") is not None
    ]

    if cross_scores:
        best_logit = max(cross_scores)
        return 1 / (1 + math.exp(-best_logit))

    scores = [
        chunk["semantic_score"]
        for chunk in retrieved_chunks
        if chunk["semantic_score"] is not None
    ]

    return max(scores) if scores else 0.0


def build_retrieval_query(question):
    """
    Prepends the previous turn's question when conversation history exists,
    so pronoun-only follow-ups ("What algorithm did he use?") have context
    to match against during retrieval. Confirmed working in the v0.6.5
    interactive session.
    """

    if conversation_history:
        last_question, _ = conversation_history[-1]
        return f"{last_question} {question}"

    return question


def build_context(retrieved_chunks, top_context=3):
    """
    v0.6.6 fix: assembles context from the top `top_context` DISTINCT
    retrieval ranks (each rank's matched chunk + its context-expansion
    neighbors), instead of positionally slicing the expanded list.

    Previously `retrieved_chunks[:top_context]` almost always returned
    only rank-1's own neighborhood (since the list is grouped
    [rank-1 + neighbors, rank-2 + neighbors, ...]), so the LLM rarely saw
    content from ranks 2-3 even when they were correctly retrieved.
    """

    grouped = {}
    ranks_seen = []

    for chunk in retrieved_chunks:
        r = chunk["retrieval_rank"]
        if r not in grouped:
            grouped[r] = []
            ranks_seen.append(r)
        grouped[r].append(chunk)

    selected_ranks = sorted(ranks_seen)[:top_context]

    pieces = []

    for r in selected_ranks:
        group = sorted(grouped[r], key=lambda c: c["chunk_id"])
        pieces.append("\n".join(c["text"] for c in group))

    return "\n\n---\n\n".join(pieces)


def answer_question(question, retrieved_chunks=None, top_context=3,
                     use_reranker=False, rerank_weight=0.6):
    """
    Single entry point for asking a question end-to-end.

    v0.6.6: context built via build_context() (top-N distinct ranks, not a
    positional slice); wrapped in try/except so a single bad query returns
    a graceful fallback instead of crashing a long session; returns
    elapsed time as a 4th value for latency visibility.
    """

    start = time.time()

    try:

        if retrieved_chunks is None:
            retrieval_query = build_retrieval_query(question)
            retrieved_chunks = retrieve(
                retrieval_query,
                use_reranker=use_reranker,
                rerank_weight=rerank_weight
            )

        confidence = assess_confidence(retrieved_chunks)

        if confidence < CONFIDENCE_THRESHOLD:
            elapsed = time.time() - start
            return DONT_KNOW_MESSAGE, confidence, retrieved_chunks, elapsed

        context = build_context(retrieved_chunks, top_context=top_context)

        answer = generate_answer(question, context)

        elapsed = time.time() - start

        return answer, confidence, retrieved_chunks, elapsed

    except Exception as e:
        elapsed = time.time() - start
        print(f"[answer_question error] {type(e).__name__}: {e}")
        return (
            "Sorry, something went wrong answering that question.",
            0.0,
            retrieved_chunks if retrieved_chunks is not None else [],
            elapsed
        )


def calibrate_confidence_threshold(use_reranker=False, rerank_weight=0.6):
    """
    Runs retrieval confidence over every question in gold_eval.json under
    the given retrieval mode, split by answerable/unanswerable. Requires
    `gold` to already be loaded.
    """

    answerable_scores = []
    unanswerable_scores = []

    for sample in gold:

        results = retrieve(
            sample["question"],
            use_reranker=use_reranker,
            rerank_weight=rerank_weight
        )
        confidence = assess_confidence(results)

        if sample["answerable"]:
            answerable_scores.append(confidence)
        else:
            unanswerable_scores.append(confidence)

        label = "ANSWERABLE" if sample["answerable"] else "UNANSWERABLE"
        print(f"[{label:>12}] {confidence:.3f}  {sample['question']}")

    print("\n" + "=" * 70)

    if answerable_scores:
        print(
            f"Answerable   -> min {min(answerable_scores):.3f}, "
            f"max {max(answerable_scores):.3f}, "
            f"avg {sum(answerable_scores)/len(answerable_scores):.3f}"
        )

    if unanswerable_scores:
        print(
            f"Unanswerable -> min {min(unanswerable_scores):.3f}, "
            f"max {max(unanswerable_scores):.3f}, "
            f"avg {sum(unanswerable_scores)/len(unanswerable_scores):.3f}"
        )

    print("=" * 70)
    print("Pick CONFIDENCE_THRESHOLD between the top of the unanswerable")
    print("range and the bottom of the answerable range. If the ranges")
    print("overlap, that's a real finding, not a bug in this function.")


def add_to_gold_set(question, expected_answer, gold_page=None, gold_chunk=None,
                     answerable=True, keywords=None, notes="", save_path="gold_eval.json"):
    """
    Capture a failure found during interactive testing straight into the
    gold set in one line.
    """

    entry = {
        "question": question,
        "expected_answer": expected_answer,
        "keywords": keywords or [],
        "gold_page": gold_page if gold_page is not None else -1,
        "gold_chunk": gold_chunk if gold_chunk is not None else -1,
        "answerable": answerable,
        "notes": notes
    }

    gold.append(entry)

    with open(save_path, "w") as f:
        json.dump(gold, f, indent=2, ensure_ascii=False)

    print(f"Added to gold set ({len(gold)} total): {question}")


def compare_retrieval_modes(rerank_weight=0.6):
    """
    Runs retrieval accuracy across hybrid-only, full reranker override, and
    blended reranking, on every ANSWERABLE gold question. Requires `gold`
    to already be loaded.
    """

    modes = {
        "hybrid_only": dict(use_reranker=False),
        "reranker_full_override": dict(use_reranker=True, rerank_weight=1.0),
        f"blended_(weight={rerank_weight})": dict(use_reranker=True, rerank_weight=rerank_weight),
    }

    answerable = [s for s in gold if s["answerable"]]

    print(f"Comparing retrieval modes on {len(answerable)} answerable gold questions...\n")

    for mode_name, kwargs in modes.items():

        correct = 0
        misses = []

        for sample in answerable:

            results = retrieve(sample["question"], **kwargs)

            matched = [c for c in results if not c["context_neighbor"]][:3]
            pages = [c["page"] for c in matched]
            chunk_ids = [c["chunk_id"] for c in matched]

            passed = (
                sample["gold_chunk"] in chunk_ids
                or sample["gold_page"] in pages
            )

            if passed:
                correct += 1
            else:
                misses.append(sample["question"])

        print(f"{mode_name:32s} {correct}/{len(answerable)}")
        if misses:
            for m in misses:
                print(f"    missed: {m}")

    print("\nPick the mode with the best score - or the fewest misses on")
    print("questions you care most about - and set use_reranker/rerank_weight")
    print("accordingly in answer_question() calls going forward.")


###Gold Evaluation Set

In [ ]:
questions = [
    "What algorithm was used for classification?",
    "How was the dataset split?",
    "Why was GBDT chosen?",
    "Which inspiral parameters were used?",
    "What does Classifier A predict?",
    "Who is the author of the report?",
    "What is SHAP used for?",
    "Which real gravitational-wave events were analysed?",
    "What optimizer was used to train the neural network?",
    "What GPU was used for training?"
]

In [ ]:
for q in questions:
    debug_retrieval(q)

QUERY : What algorithm was used for classification?

RETRIEVAL RANK 1

Context Chunk
Document : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page     : 8
Chunk ID : 33
------------------------------------------------------------------------------------------
difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more realistic and stable performance estimates.


Matched Chunk
Semantic Score : 0.6347
BM25 Score     : 13.7764
RRF Score      : 0.03

In [ ]:
import json

Load the gold set (v0.6.6 revision - updated `notes` reflecting this session's findings).

In [ ]:
with open("gold_eval.json") as f:
    gold = json.load(f)

print(f"Loaded {len(gold)} gold questions.")


Loaded 10 gold questions.


Calibrate against the default hybrid-only retrieval, before interactive testing.

In [ ]:
calibrate_confidence_threshold()


[  ANSWERABLE] 0.635  What algorithm was used for classification?
[  ANSWERABLE] 0.539  How was the dataset split?
[  ANSWERABLE] 0.588  Why was GBDT chosen?
[  ANSWERABLE] 0.411  Which inspiral parameters were used?
[  ANSWERABLE] 0.472  What does Classifier A predict?
[  ANSWERABLE] 0.724  Who is the author of Machine Learning Classification of Binary Neutron Star Remnants Using Gravitational Wave Data?
[  ANSWERABLE] 0.475  What is SHAP used for?
[  ANSWERABLE] 0.527  Which real gravitational-wave events were analysed?
[UNANSWERABLE] 0.277  What optimizer was used to train the neural network?
[UNANSWERABLE] 0.301  What GPU was used for training?

Answerable   -> min 0.411, max 0.724, avg 0.546
Unanswerable -> min 0.277, max 0.301, avg 0.289
Pick CONFIDENCE_THRESHOLD between the top of the unanswerable
range and the bottom of the answerable range. If the ranges
overlap, that's a real finding, not a bug in this function.


Compare retrieval modes now that gold is loaded.

In [ ]:
compare_retrieval_modes()


Comparing retrieval modes on 8 answerable gold questions...

hybrid_only                      7/8
    missed: Which real gravitational-wave events were analysed?
reranker_full_override           7/8
    missed: What algorithm was used for classification?
blended_(weight=0.6)             7/8
    missed: What algorithm was used for classification?

Pick the mode with the best score - or the fewest misses on
questions you care most about - and set use_reranker/rerank_weight
accordingly in answer_question() calls going forward.


## Interactive chat

**v0.6.6 changes:** prints elapsed time alongside confidence; skips empty
input instead of erroring; wraps each turn in try/except so one bad query
doesn't end the whole session. Retest "What algorithm did he use?" as a
follow-up, and "What does Classifier A predict?" - both should be more
consistent now that context assembly actually includes multiple retrieved
ranks.

In [ ]:
while True:
    print("\n" + "=" * 80)

    question = input("\nAsk a question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("\nGoodbye!")
        break

    if not question.strip():
        print("(empty question, try again)")
        continue

    try:
        answer, confidence, _, elapsed = answer_question(question)

        conversation_history.append((question, answer))

        print("\n" + "=" * 80)
        print(f"Confidence: {confidence:.3f}  |  {elapsed:.2f}s")
        print(answer)

    except Exception as e:
        print(f"Something went wrong: {type(e).__name__}: {e}")
        print("Try again or type 'exit' to quit.")




Confidence: 0.724  |  832.40s
Surendaranath Kannyyappan


Confidence: 0.684  |  923.74s
Gradient-Boosted Decision Trees were used.


Confidence: 0.488  |  922.40s
I could not find that information in the provided document.



## Known Limitations (v0.6.6)

- **Still single-document.** This is the one remaining item - the
  different-PDF test - before v0.7.
- **The Classifier A/B/C class-list content lives across several small,
  fragmented chunks** (one chunk, "granularity:", is nearly empty due to a
  heading-detection heuristic misfiring on a short list-intro line).
  `build_context()` mitigates this by pulling in more distinct retrieval
  ranks, but the underlying chunking artifact isn't fixed - worth revisiting
  if this specific question still underperforms after retesting.
- **Author-name spelling needs a decision** based on what the diagnostic
  cell shows - either a PDF-extraction issue (fixable by cleaning that
  chunk's text) or a generation quirk (harder to fully eliminate, but the
  no-invention prompt rule should keep it rare).
- **Reranking's real value on this document is proven negative on this
  gold set** (`compare_retrieval_modes()`: 7/8 hybrid vs 6/8 full-rerank vs
  7/8 blended) - kept available but off by default.
- **Generation-correctness beyond retrieval is still unchecked.** No
  automated check confirms the generated text is factually correct given
  good context - this remains a v0.7+ item (answer verification), not
  something in scope for the notebook stage.


Demo: out-of-domain question, still expected to trigger the confidence gate.

In [ ]:
query = "Which sport is most played?"

answer, confidence, retrieved_chunks, elapsed = answer_question(query)

print("=" * 80)
print("QUESTION")
print(query)

print("\n" + "=" * 80)
print(f"Confidence: {confidence:.3f}  |  {elapsed:.2f}s")
print("ANSWER")
print(answer)


## Evaluation functions

`evaluate_retrieval` keeps the earlier fix (top-k distinct matched chunks).
`evaluate_generation` now unpacks the 4-tuple and prints elapsed time.
`evaluate_answer` keeps LaTeX/unicode-robust keyword matching.

In [ ]:
def evaluate_retrieval(retrieved_chunks, sample, top_k=3):

    if not sample["answerable"]:
        return True

    matched = [
        chunk for chunk in retrieved_chunks
        if not chunk["context_neighbor"]
    ][:top_k]

    retrieved_pages = [chunk["page"] for chunk in matched]
    retrieved_chunk_ids = [chunk["chunk_id"] for chunk in matched]

    passed = (
        sample["gold_chunk"] in retrieved_chunk_ids
        or
        sample["gold_page"] in retrieved_pages
    )

    print("Retrieval :", "PASS" if passed else "FAIL")
    print("Expected Page :", sample["gold_page"])
    print("Retrieved Pages :", retrieved_pages)

    return passed


In [ ]:
def evaluate_generation(sample, retrieved_chunks):

    answer, confidence, _, elapsed = answer_question(
        sample["question"],
        retrieved_chunks=retrieved_chunks
    )

    print(f"\nRetrieval Confidence: {confidence:.3f}  |  Elapsed: {elapsed:.2f}s")

    print("\nGenerated Answer\n")
    print(answer)

    return answer


**v0.6.5 fix:** normalizes LaTeX commands (`\\chi` -> `chi`, `\\text{eff}` -> `eff`)
and unicode Greek letters (`χ` -> `chi`, `λ`/`Λ` -> `lambda`) before keyword
comparison, so a correct answer formatted as `\\chi_{\\text{eff}}` still
matches a plain-text gold keyword like "χeff".

In [ ]:
import re

_LATEX_TEXT_RE = re.compile(r'\\text\{([^}]*)\}')
_LATEX_CMD_RE = re.compile(r'\\([a-zA-Z]+)')
_UNICODE_MAP = {
    "χ": "chi",
    "λ": "lambda",
    "̃": "",
    "~": "",
}


def normalize_for_match(text):

    text = text.lower()

    text = _LATEX_TEXT_RE.sub(r'\1', text)
    text = _LATEX_CMD_RE.sub(r'\1', text)

    for old, new in _UNICODE_MAP.items():
        text = text.replace(old, new)

    for ch in ["{", "}", "_", "(", ")", "$", "\\"]:
        text = text.replace(ch, "")

    text = re.sub(r'\s+', '', text)

    return text


def evaluate_answer(answer, sample):

    if not sample["answerable"]:

        passed = (
            "could not find" in answer.lower()
            or
            "not found" in answer.lower()
        )

    else:

        norm_answer = normalize_for_match(answer)

        hits = sum(
            normalize_for_match(keyword) in norm_answer
            for keyword in sample["keywords"]
        )

        passed = (
            hits / len(sample["keywords"])
        ) >= 0.65

    print("\nExpected Answer\n")
    print(sample["expected_answer"])

    print("\nGeneration :", "PASS" if passed else "FAIL")

    return passed


In [ ]:
def evaluate_rag():

    retrieval_correct = 0
    generation_correct = 0

    total = len(gold)

    for sample in gold:

        print("=" * 80)
        print("QUESTION")
        print(sample["question"])
        print("=" * 80)

        retrieved_chunks = retrieve(sample["question"])

        retrieval_pass = evaluate_retrieval(
            retrieved_chunks,
            sample
        )

        if retrieval_pass:
            retrieval_correct += 1

        answer = evaluate_generation(
            sample,
            retrieved_chunks
        )

        generation_pass = evaluate_answer(
            answer,
            sample
        )

        if generation_pass:
            generation_correct += 1

    print("\n" + "=" * 80)

    print(f"Retrieval Accuracy : {retrieval_correct}/{total} ({100*retrieval_correct/total:.1f}%)")
    print(f"Generation Accuracy: {generation_correct}/{total} ({100*generation_correct/total:.1f}%)")

In [ ]:
def evaluate_subset(indices):

    retrieval_correct = 0
    generation_correct = 0

    for idx in indices:

        sample = gold[idx]

        print("=" * 80)
        print("QUESTION")
        print(sample["question"])
        print("=" * 80)

        retrieved_chunks = retrieve(sample["question"])

        retrieval_pass = evaluate_retrieval(
            retrieved_chunks,
            sample
        )

        if retrieval_pass:
            retrieval_correct += 1

        answer = evaluate_generation(
            sample,
            retrieved_chunks
        )

        generation_pass = evaluate_answer(
            answer,
            sample
        )

        if generation_pass:
            generation_correct += 1

    print("\n" + "=" * 80)

    print(
        f"Retrieval Accuracy : {retrieval_correct}/{len(indices)} "
        f"({100*retrieval_correct/len(indices):.1f}%)"
    )

    print(
        f"Generation Accuracy: {generation_correct}/{len(indices)} "
        f"({100*generation_correct/len(indices):.1f}%)"
    )

In [ ]:

evaluate_subset([6])

In [ ]:
evaluate_rag()